# RNN on CSV Upload Time Series

## Objective
Train SimpleRNN using a single numeric column from uploaded CSV time series data.

## Google Colab Steps
1. Run the setup cell.
2. Execute the main cell.
3. Save a copy to your Google Drive if needed.
## Notes
The first numeric column is treated as a univariate time series.



In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
display(df.head())

numeric_df = df.select_dtypes(include=np.number).dropna()
if numeric_df.shape[1] < 1:
    raise ValueError("CSV must contain at least one numeric column for time series.")

series = numeric_df.iloc[:, 0].to_numpy(dtype=np.float32)
series = (series - series.mean()) / (series.std() + 1e-8)

window = 20
X, y = [], []
for i in range(len(series) - window):
    X.append(series[i:i + window])
    y.append(series[i + window])

X = np.array(X)[..., np.newaxis]
y = np.array(y)

if len(X) < 100:
    raise ValueError("Need more rows in CSV to create RNN training sequences.")

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = models.Sequential([
    layers.SimpleRNN(32, input_shape=(window, 1)),
    layers.Dense(1),
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(X_train, y_train, epochs=8, batch_size=32, validation_data=(X_test, y_test), verbose=1)

pred = model.predict(X_test, verbose=0).ravel()

plt.figure(figsize=(8, 4))
plt.plot(y_test[:200], label="Actual")
plt.plot(pred[:200], label="Predicted")
plt.title("RNN Forecast on Uploaded CSV Series")
plt.legend()
plt.show()
